# AIG230 - Assignment 6: Natural Language Processing

In [24]:
import random
from collections import Counter
import nltk
from nltk.corpus import brown

nltk.download("brown", quiet=True)

BOS = "<bos>"
EOS = "<eos>"
UNK = "<unk>"
SPECIAL_TOKENS = [UNK, BOS, EOS]
SEED = 42
SPLIT_RATIOS = (0.8, 0.1, 0.1)  # train, valid, test
MIN_FREQ = 2

def is_punctuation_only(token: str) -> bool:
    return not any(ch.isalnum() for ch in token)


def preprocess_sentence(sentence_tokens):
    cleaned = []
    for token in sentence_tokens:
        tok = token.lower()
        if is_punctuation_only(tok):
            continue
        cleaned.append(tok)
    return [BOS] + cleaned + [EOS]


raw_sentences = brown.sents(categories="news")
processed_sentences = [preprocess_sentence(sent) for sent in raw_sentences]

# Remove degenerate samples like [<bos>, <eos>] after punctuation filtering
processed_sentences = [s for s in processed_sentences if len(s) > 2]

In [25]:
# ----- D1: One shared split (sentence-level, deterministic) -----
rng = random.Random(SEED)
indices = list(range(len(processed_sentences)))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(SPLIT_RATIOS[0] * n_total)
n_valid = int(SPLIT_RATIOS[1] * n_total)

train_idx = indices[:n_train]
valid_idx = indices[n_train:n_train + n_valid]
test_idx = indices[n_train + n_valid:]

train_sentences = [processed_sentences[i] for i in train_idx]
valid_sentences = [processed_sentences[i] for i in valid_idx]
test_sentences = [processed_sentences[i] for i in test_idx]

train_tokens_raw = [tok for sent in train_sentences for tok in sent]
valid_tokens_raw = [tok for sent in valid_sentences for tok in sent]
test_tokens_raw = [tok for sent in test_sentences for tok in sent]

In [26]:
# ----- D2: Vocabulary from training split only (shared) -----
train_counter = Counter(train_tokens_raw)

vocab = list(SPECIAL_TOKENS)
for token, freq in train_counter.items():
    if token in {UNK, BOS, EOS}:
        continue
    if freq >= MIN_FREQ:
        vocab.append(token)

token_to_id = {tok: i for i, tok in enumerate(vocab)}
id_to_token = {i: tok for tok, i in token_to_id.items()}


def map_sentence_to_vocab(sentence):
    return [tok if tok in token_to_id else UNK for tok in sentence]


train_sentences_mapped = [map_sentence_to_vocab(s) for s in train_sentences]
valid_sentences_mapped = [map_sentence_to_vocab(s) for s in valid_sentences]
test_sentences_mapped = [map_sentence_to_vocab(s) for s in test_sentences]

train_tokens = [tok for sent in train_sentences_mapped for tok in sent]
valid_tokens = [tok for sent in valid_sentences_mapped for tok in sent]
test_tokens = [tok for sent in test_sentences_mapped for tok in sent]

def tokens_to_ids(tokens):
    return [token_to_id.get(tok, token_to_id[UNK]) for tok in tokens]


train_ids = tokens_to_ids(train_tokens)
valid_ids = tokens_to_ids(valid_tokens)
test_ids = tokens_to_ids(test_tokens)

train_sentence_ids = [tokens_to_ids(s) for s in train_sentences_mapped]
valid_sentence_ids = [tokens_to_ids(s) for s in valid_sentences_mapped]
test_sentence_ids = [tokens_to_ids(s) for s in test_sentences_mapped]

In [ ]:
# ----- Reports (D1 + D2) -----
print("D1 Report")
print(f"Sentences (train/valid/test): {len(train_sentences)} / {len(valid_sentences)} / {len(test_sentences)}")
print(f"Tokens    (train/valid/test): {len(train_tokens_raw)} / {len(valid_tokens_raw)} / {len(test_tokens_raw)}")

print(f"Vocab size (train-only): {len(vocab)}")
print()

print("D2 Report")
print(f"Chosen min_freq: {MIN_FREQ}")
print(f"Final vocab size: {len(vocab)}")
print("Top 20 most frequent tokens in train split:")
for tok, freq in train_counter.most_common(20):
    print(f"  {tok:>12} : {freq}")
print()

print(f"Special token IDs: UNK={token_to_id[UNK]}, BOS={token_to_id[BOS]}, EOS={token_to_id[EOS]}")
print("Example processed sentence:", train_sentences_mapped[0][:20])

D1 Report
Sentences (train/valid/test): 3688 / 461 / 462
Tokens    (train/valid/test): 77883 / 10104 / 9827
Vocab size (train-only): 5396

D2 Report
Chosen min_freq: 2
Final vocab size: 5396
Top 20 most frequent tokens in train split:
           the : 5073
         <bos> : 3688
         <eos> : 3688
            of : 2288
           and : 1734
             a : 1690
            to : 1682
            in : 1581
           for : 760
          that : 653
            is : 576
            on : 556
           was : 553
            he : 517
            at : 500
          with : 471
            as : 411
            be : 403
            by : 402
            it : 379

Special token IDs: UNK=0, BOS=1, EOS=2
Example processed sentence: ['<bos>', 'the', 'land', 'lies', 'ready', 'for', 'the', 'coming', '<unk>', '<eos>']


## Part A – Statistical n-gram Language Model

### A1. Train a Trigram Language Model

In [32]:
from nltk import ngrams
from nltk.lm.models import Lidstone
from nltk.lm.preprocessing import pad_both_ends, padded_everygram_pipeline

NGRAM_ORDER = 3
SMOOTHING_NAME = "Lidstone (add-k)"
LIDSTONE_GAMMA = 0.1

train_sentences_lm = train_sentences_mapped
valid_sentences_lm = valid_sentences_mapped
test_sentences_lm = test_sentences_mapped

train_data, padded_vocab = padded_everygram_pipeline(NGRAM_ORDER, train_sentences_lm)
lm = Lidstone(gamma=LIDSTONE_GAMMA, order=NGRAM_ORDER)
lm.fit(train_data, padded_vocab)

print("A1 Report")
print(f"Model: {NGRAM_ORDER}-gram language model")
print(f"Smoothing: {SMOOTHING_NAME}, k={LIDSTONE_GAMMA}")

A1 Report
Model: 3-gram language model
Smoothing: Lidstone (add-k), k=0.1


### A1 comments

- model used: trigram language model (`n=3`) with lidstone smoothing (`k=0.1`).
- why smoothing is necessary: without smoothing, unseen trigrams receive zero probability, which can collapse sentence probability and inflate perplexity.
- with lidstone smoothing, unseen events get small non-zero mass, making evaluation and generation more robust.

### A2. Evaluate with Perplexity

In [33]:
def sentence_trigrams(sentence):
    padded_sentence = list(pad_both_ends(sentence, n=NGRAM_ORDER))
    return list(ngrams(padded_sentence, NGRAM_ORDER))


def corpus_perplexity(model, sentences):
    all_trigrams = []
    for sentence in sentences:
        all_trigrams.extend(sentence_trigrams(sentence))
    return model.perplexity(all_trigrams)


valid_perplexity = corpus_perplexity(lm, valid_sentences_lm)
test_perplexity = corpus_perplexity(lm, test_sentences_lm)

print("A2 Report")
print(f"Validation perplexity: {valid_perplexity:.3f}")
print(f"Test perplexity:       {test_perplexity:.3f}")

A2 Report
Validation perplexity: 1037.531
Test perplexity:       1005.928


### A2 comments

based on the latest results (`validation=1037.531`, `test=1005.928`):

- test perplexity is slightly lower than validation perplexity, suggesting comparable generalization across held-out splits.
- these values are relatively high, which is common for sparse trigram models on open-vocabulary news text with many rare contexts.
- key factors affecting perplexity here are training size/domain match, n-gram order, smoothing strength (`k`), and unknown-token handling.

### A4. Text Generation

In [31]:
def generate_sample(model, min_tokens=30, seed=0):
    cleaned = []
    attempts = 0
    while len(cleaned) < min_tokens and attempts < 10:
        generated = model.generate(200, random_seed=seed + attempts * 17)
        cleaned.extend([tok for tok in generated if tok not in {"<s>", "</s>", BOS, EOS}])
        attempts += 1
    if len(cleaned) < min_tokens:
        cleaned.extend([UNK] * (min_tokens - len(cleaned)))
    return cleaned[:min_tokens]


print("A4 Report")
for i in range(3):
    sample_tokens = generate_sample(lm, min_tokens=30, seed=SEED + i)
    print(f"Sample {i + 1} ({len(sample_tokens)} tokens):")
    print(" ".join(sample_tokens))
    print()

A4 Report
Sample 1 (30 tokens):
library <unk> stram and was a dead issue tie dinner wednesday may 3 in the <unk> <unk> took his <unk> about himself for the <unk> fiction <unk> knows what it

Sample 2 (30 tokens):
<unk> is expected to give it the short-term advantage which it began the series is bob friend who was sentenced to 15 years in high school seven cases had a

Sample 3 (30 tokens):
beauty and three <unk> <unk> enough signatures to assure the orderly and effective conduct of the national maintenance company at <unk> eight hits for two days with the wide distribution



### A4 comments on generated samples

based on the latest three samples above:

- grammaticality: some local chunks read plausibly (for example, "is expected to give it the short-term advantage"), but several sequences are still awkward.
- coherence: each sample keeps short local flow, but global sentence-level meaning is weak and often drifts.
- repetition/noise: `<unk>` appears frequently across all three samples, which reduces readability.
- limitation summary: this trigram model captures nearby token transitions well but cannot reliably maintain long-range coherence with only two-token context.